In [4]:
import numpy as np
from sklearn.cluster import KMeans
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import SGD
import tensorflow as tf
from scipy.sparse import load_npz

# Configure GPU settings
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Restrict TensorFlow to only use the first GPU
        tf.config.set_visible_devices(gpus[0], 'GPU')
        # Allow memory growth to avoid allocating all GPU memory at once
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print(f"Using GPU: {gpus[0]}")
    except RuntimeError as e:
        print(e)

# Load your preprocessed data
X = load_npz("data_prepared_final.npz").toarray()

# Parameters
n_clusters = 10  # Adjust based on your needs
batch_size = 512  # Increased for GPU efficiency
pretrain_epochs = 100
maxiter = 1000
tol = 0.001
update_interval = 10

# Autoencoder Architecture
input_dim = X.shape[1]
encoding_dim = 500
hidden_dim = 500

# Build autoencoder inside GPU context
with tf.device('/GPU:0'):
    input_layer = Input(shape=(input_dim,))
    encoder = Dense(encoding_dim, activation='relu')(input_layer)
    encoder = Dense(hidden_dim, activation='relu')(encoder)
    decoder = Dense(encoding_dim, activation='relu')(encoder)
    decoder = Dense(input_dim)(decoder)
    autoencoder = Model(inputs=input_layer, outputs=decoder)

# Pretrain the Autoencoder on GPU
def pretrain_autoencoder():
    with tf.device('/GPU:0'):
        autoencoder.compile(optimizer='adam', loss='mse')
        autoencoder.fit(X, X, 
                       batch_size=batch_size, 
                       epochs=pretrain_epochs,
                       verbose=1)
    return Model(inputs=input_layer, outputs=encoder)

pretrain_autoencoder()

# Build the Encoder Model on GPU
with tf.device('/GPU:0'):
    encoder_model = Model(inputs=input_layer, outputs=encoder)
    encoded_X = encoder_model.predict(X, batch_size=batch_size)

# Initialize Cluster Centers with k-means (on CPU)
kmeans = KMeans(n_clusters=n_clusters, n_init=20)
y_pred = kmeans.fit_predict(encoded_X)

# Define Clustering Layer with GPU-optimized operations
class ClusteringLayer(tf.keras.layers.Layer):
    def __init__(self, n_clusters, weights=None, alpha=1.0, **kwargs):
        super(ClusteringLayer, self).__init__(**kwargs)
        self.n_clusters = n_clusters
        self.alpha = alpha
        self.initial_weights = weights

    def build(self, input_shape):
        input_dim = input_shape[1]
        self.clusters = self.add_weight(
            shape=(self.n_clusters, input_dim),
            initializer='glorot_uniform',
            name='clusters'
        )
        if self.initial_weights is not None:
            self.set_weights(self.initial_weights)
            del self.initial_weights
        self.built = True

    def call(self, inputs, **kwargs):
        # GPU-optimized operations
        expanded = tf.expand_dims(inputs, axis=1)
        squared_diff = tf.square(expanded - self.clusters)
        sum_squared = tf.reduce_sum(squared_diff, axis=2)
        q = 1.0 / (1.0 + (sum_squared / self.alpha))
        q = q ** ((self.alpha + 1.0) / 2.0)
        q = tf.transpose(tf.transpose(q) / tf.reduce_sum(q, axis=1))
        return q

# Build DEC Model on GPU
with tf.device('/GPU:0'):
    clustering_layer = ClusteringLayer(n_clusters, name='clustering')(encoder)
    dec_model = Model(inputs=input_layer, outputs=clustering_layer)
    
    # Initialize cluster centers from k-means
    dec_model.get_layer(name='clustering').set_weights([kmeans.cluster_centers_])

# Define Target Distribution
def target_distribution(q):
    weight = q ** 2 / q.sum(0)
    return (weight.T / weight.sum(1)).T

# Train DEC Model on GPU
with tf.device('/GPU:0'):
    dec_model.compile(optimizer=SGD(0.01, 0.9), loss='kld')
    
    # Initialize training variables
    index_array = np.arange(X.shape[0])
    y_pred_last = np.copy(y_pred)
    index = 0
    
    for ite in range(int(maxiter)):
        if ite % update_interval == 0:
            q = dec_model.predict(X, batch_size=batch_size, verbose=0)
            p = target_distribution(q)
            y_pred = q.argmax(1)
            
            # Check stop criterion
            if ite > 0:
                delta_label = np.sum(y_pred != y_pred_last).astype(np.float32) / y_pred.shape[0]
                y_pred_last = np.copy(y_pred)
                if delta_label < tol:
                    print(f'Reached tolerance threshold. Stopping training.')
                    break
        
        # Train on batch
        idx = index_array[index * batch_size: min((index+1) * batch_size, X.shape[0])]
        dec_model.train_on_batch(x=X[idx], y=p[idx])
        index = index + 1 if (index + 1) * batch_size <= X.shape[0] else 0

# Final clustering on GPU
with tf.device('/GPU:0'):
    q = dec_model.predict(X, batch_size=batch_size, verbose=0)
    y_pred = q.argmax(1)

Epoch 1/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 9s 235ms/step - loss: 7.4909e-04
Epoch 2/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 219ms/step - loss: 5.7007e-04
Epoch 3/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 219ms/step - loss: 4.4325e-04
Epoch 4/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 223ms/step - loss: 4.2353e-04
Epoch 5/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 218ms/step - loss: 3.5784e-04
Epoch 6/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 218ms/step - loss: 3.1297e-04
Epoch 7/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 221ms/step - loss: 3.9670e-04
Epoch 8/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 221ms/step - loss: 3.4016e-04
Epoch 9/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 218ms/step - loss: 2.8956e-04
Epoch 10/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 212ms/step - loss: 2.7643e-04
Epoch 11/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 211ms/step - loss: 2.6903e-04
Epoch 12/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 214ms/step - loss: 2.6149e-04
Epoch 13/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 218ms/step - loss: 2.5944e-04
Epoch 14/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 214ms/step - loss

In [5]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# Calculate evaluation metrics
silhouette = silhouette_score(X, y_pred)
calinski = calinski_harabasz_score(X, y_pred)
davies = davies_bouldin_score(X, y_pred)

print(f"Silhouette Score: {silhouette:.3f} (Higher is better, range [-1, 1])")
print(f"Calinski-Harabasz Score: {calinski:.3f} (Higher is better)")
print(f"Davies-Bouldin Score: {davies:.3f} (Lower is better)")

Silhouette Score: 0.042 (Higher is better, range [-1, 1])
Calinski-Harabasz Score: 910.465 (Higher is better)
Davies-Bouldin Score: 4.237 (Lower is better)
